cell 0

In [3]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

# ── قراءة الـ features بس ──────────────────────────────────
df = pq.read_table(
    'df_balanced_C.parquet'
).to_pandas()

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df['Label'].value_counts())

Shape: (1971791, 81)

Columns: ['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag C

CELL 1 — Imports & Config

In [9]:
# ╔══════════════════════════════════════════════════════════╗
# ║      CELL 1 — Imports & Config                           ║
# ╚══════════════════════════════════════════════════════════╝
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import joblib
import gc
import pyarrow.parquet as pq
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, BatchNormalization,
    Activation, Dropout
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

MODEL_PATH = '/mnt/d/DATASETS/IDS/MLP_Model/'
os.makedirs(MODEL_PATH, exist_ok=True)

print(f"TensorFlow : {tf.__version__}")
print(f"Model Path : {MODEL_PATH}")
print("✅ Imports Done")

TensorFlow : 2.21.0
Model Path : /mnt/d/DATASETS/IDS/MLP_Model/
✅ Imports Done


CELL 2 — Load & Filter Data

In [10]:
# ╔══════════════════════════════════════════════════════════╗
# ║      CELL 2 — Load & Filter Data                         ║
# ╚══════════════════════════════════════════════════════════╝

# ── Classes اللي هنشيلها ────────────────────────────────────
DROP_LABELS = [
    'Brute Force -Web',
    'Brute Force -XSS',
    'DDoS attack-LOIC-UDP',
    'Telnet-Bruteforce' ,
    'Infiltration'
]

# ── Columns اللي مش features ────────────────────────────────
DROP_COLS = ['Timestamp', 'source']

print("Loading data...")
df = pq.read_table(
    '/mnt/d/DATASETS/IDS/Cleaning_data/df_balanced_C.parquet'
).to_pandas()

print(f"Before filter: {df.shape}")

# ── شيل الـ classes الضعيفة ──────────────────────────────────
df = df[~df['Label'].isin(DROP_LABELS)].reset_index(drop=True)

# ── شيل الـ columns غير المفيدة ──────────────────────────────
df.drop(columns=DROP_COLS, inplace=True)

print(f"After filter : {df.shape}")
print(f"\n=== Label Distribution ===")
for label, count in df['Label'].value_counts().items():
    pct = count / len(df) * 100
    print(f"  {label:<30} : {count:>8,} ({pct:.1f}%)")

print("\n✅ Data Loaded & Filtered")

Loading data...
Before filter: (1971791, 81)
After filter : (1793354, 79)

=== Label Distribution ===
  Benign                         :  200,000 (11.2%)
  Bot                            :  200,000 (11.2%)
  DDoS attack-HOIC               :  200,000 (11.2%)
  Reconnaissance                 :  200,000 (11.2%)
  SQL Injection                  :  200,000 (11.2%)
  DoS attacks                    :  200,000 (11.2%)
  DDoS attacks-LOIC-HTTP         :  200,000 (11.2%)
  SSH-Bruteforce                 :  200,000 (11.2%)
  FTP-BruteForce                 :  193,354 (10.8%)

✅ Data Loaded & Filtered


CELL 3 — Feature Engineering & Cleaning

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import numpy as np

FEATURE_COLS = [c for c in df.columns if c not in ['Label', 'Timestamp', 'source']]
X = df[FEATURE_COLS].values.astype('float32')

le_temp = LabelEncoder()
y_temp  = le_temp.fit_transform(df['Label'].values)

# ── نستخدم subset عشان يكون أسرع ──────────────────────────── 
idx     = np.random.choice(len(X), 200_000, replace=False)
X_sub   = X[idx]
y_sub   = y_temp[idx]

print("Training Random Forest for feature importance...")
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_sub, y_sub)

# ── Feature Importance ─────────────────────────────────────── 
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS)
importances = importances.sort_values(ascending=False)

print(f"\n=== Feature Importance (Top 20) ===")
for feat, imp in importances.head(20).items():
    bar = '█' * int(imp * 500)
    print(f"  {feat:<35} : {imp:.4f} {bar}")

print(f"\n=== Low Importance (<0.001) ===")
low_imp = importances[importances < 0.001]
print(f"Count: {len(low_imp)}")
for feat, imp in low_imp.items():
    print(f"  {feat:<35} : {imp:.4f}")

print(f"\nRF Accuracy: {rf.score(X_sub, y_sub)*100:.2f}%")

Training Random Forest for feature importance...

=== Feature Importance (Top 20) ===
  Dst Port                            : 0.0913 █████████████████████████████████████████████
  Fwd Seg Size Min                    : 0.0651 ████████████████████████████████
  FWD Init Win Bytes                  : 0.0601 ██████████████████████████████
  Bwd Init Win Bytes                  : 0.0593 █████████████████████████████
  Flow IAT Max                        : 0.0436 █████████████████████
  SYN Flag Count                      : 0.0428 █████████████████████
  Flow Duration                       : 0.0387 ███████████████████
  Fwd Header Length                   : 0.0380 ███████████████████
  Flow Packets/s                      : 0.0360 ██████████████████
  Flow IAT Mean                       : 0.0343 █████████████████
  Bwd Packets/s                       : 0.0320 ████████████████
  ACK Flag Count                      : 0.0260 █████████████
  Subflow Fwd Packets                 : 0.0234 ███████████

# ║      CELL 5 — Drop Low Importance Features               ║

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║      CELL 5 — Drop Low Importance Features               ║
# ╚══════════════════════════════════════════════════════════╝

LOW_IMPORTANCE = [
    'Fwd PSH Flags', 'Active Mean', 'Active Max', 'Active Min',
    'Idle Std', 'Active Std', 'Fwd URG Flags', 'CWR Flag Count',
    'Bwd URG Flags', 'Bwd PSH Flags', 'Fwd Bulk Rate Avg',
    'Fwd Bytes/Bulk Avg', 'Fwd Packet/Bulk Avg'
]

# شيل بس اللي موجودة فعلاً في الداتا
to_drop = [c for c in LOW_IMPORTANCE if c in df.columns]
df.drop(columns=to_drop, inplace=True)

print(f"Dropped  : {len(to_drop)} features")
print(f"Remained : {df.shape[1] - 1} features + Label")
print(f"New shape: {df.shape}")
print("\n✅ Features Dropped")



Dropped  : 13 features 



Remained : 65 features + Label




New shape: (1793354, 66)

✅ Features Dropped



# ║      CELL 6— Scale + Encode + Split + Save              ║


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║      CELL 6— Scale + Encode + Split + Save              ║
# ╚══════════════════════════════════════════════════════════╝

FEATURE_COLS = [c for c in df.columns if c not in ['Label']]

X = df[FEATURE_COLS].values.astype('float32')
y = df['Label'].values

# ── تحقق من NaN و Inf ────────────────────────────────────
mask = ~(np.isnan(X).any(axis=1) | np.isinf(X).any(axis=1))
X = X[mask]
y = y[mask]
print(f"Samples after cleaning: {len(X):,}")

# ── Label Encoding ───────────────────────────────────────
le = LabelEncoder()
y_encoded = le.fit_transform(y)
joblib.dump(le, MODEL_PATH + 'label_encoder.pkl')
print("\n=== Classes ===")
for i, cls in enumerate(le.classes_):
    print(f"  {i:>2} → {cls}")

# ── Scaling ──────────────────────────────────────────────
print("\nScaling...")
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X).astype('float32')
X_scaled = np.clip(X_scaled, -10, 10)
joblib.dump(scaler, MODEL_PATH + 'scaler.pkl')
print(f"min={X_scaled.min():.2f}, max={X_scaled.max():.2f}")

# ── Split 70/15/15 ───────────────────────────────────────
X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.15, random_state=42, stratify=y_encoded
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.15/0.85, random_state=42, stratify=y_temp
)

del X, X_scaled, X_temp
gc.collect()

# ── Save ─────────────────────────────────────────────────
np.save(MODEL_PATH + 'X_train.npy', X_train)
np.save(MODEL_PATH + 'X_val.npy',   X_val)
np.save(MODEL_PATH + 'X_test.npy',  X_test)
np.save(MODEL_PATH + 'y_train.npy', y_train)
np.save(MODEL_PATH + 'y_val.npy',   y_val)
np.save(MODEL_PATH + 'y_test.npy',  y_test)
pd.Series(FEATURE_COLS).to_csv(MODEL_PATH + 'feature_names.csv', index=False)

n_classes   = len(le.classes_)
n_features  = X_train.shape[1]

print(f"\n=== Split ===")
print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Classes : {n_classes}")
print(f"Features: {n_features}")
print("\n Data Ready")

Samples after cleaning: 1,793,354



=== Classes ===


   0 → Benign

   
   1 → Bot

   
   2 → DDoS attack-HOIC


   
   3 → DDoS attacks-LOIC-HTTP

   
   4 → DoS attacks

   
   5 → FTP-BruteForce

   
   6 → Reconnaissance

   
   7 → SQL Injection

   
   8 → SSH-Bruteforce


Scaling...



min=-6.00, max=10.00



=== Split ===


X_train : (1255347, 65)


X_val   : (269003, 65)


X_test  : (269004, 65)


Classes : 9


Features: 65

 Data Ready

# ║      CELL 7 — Class Weights                              ║


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║      CELL 7 — Class Weights                              ║
# ╚══════════════════════════════════════════════════════════╝
from sklearn.utils.class_weight import compute_class_weight

classes_arr = np.unique(y_train)
weights     = compute_class_weight(
                  class_weight='balanced',
                  classes=classes_arr,
                  y=y_train
              )
class_weight_dict = dict(zip(classes_arr, weights))

print("=== Class Weights ===")
for idx, w in class_weight_dict.items():
    print(f"  {le.classes_[idx]:<30} : {w:.4f}")

=== Class Weights ===
  Benign                         : 0.9963
  Bot                            : 0.9963
  DDoS attack-HOIC               : 0.9963
  DDoS attacks-LOIC-HTTP         : 0.9963
  DoS attacks                    : 0.9963
  FTP-BruteForce                 : 1.0306
  Reconnaissance                 : 0.9963
  SQL Injection                  : 0.9963
  SSH-Bruteforce                 : 0.9963



# ║      CELL 8 — Build CNN+BiLSTM+Attention V2             ║


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║      CELL 8 — Build CNN+BiLSTM+Attention V2             ║
# ╚══════════════════════════════════════════════════════════╝
import matplotlib.patches as mpatches

MODEL_PATH_V2 = '/kaggle/working/CNN_Model_V2/'
os.makedirs(MODEL_PATH_V2, exist_ok=True)

def build_model_v2(n_features, n_classes):
    inputs = Input(shape=(n_features, 1), name='input')

    # CNN Block
    x = Conv1D(128, kernel_size=3, padding='same', name='conv1')(inputs)
    x = BatchNormalization(name='bn1')(x)
    x = Activation('relu', name='relu1')(x)
    x = Dropout(0.2, name='drop1')(x)

    x = Conv1D(256, kernel_size=3, padding='same', name='conv2')(x)
    x = BatchNormalization(name='bn2')(x)
    x = Activation('relu', name='relu2')(x)
    x = Dropout(0.2, name='drop2')(x)

    x = MaxPooling1D(pool_size=2, name='pool1')(x)

    # BiLSTM Block
    x = Bidirectional(LSTM(256, return_sequences=True), name='bilstm1')(x)
    x = Dropout(0.3, name='drop3')(x)

    x = Bidirectional(LSTM(128, return_sequences=True), name='bilstm2')(x)
    x = Dropout(0.3, name='drop4')(x)

    # Attention Block
    x = attention_block(x)
    x = GlobalAveragePooling1D(name='gap')(x)

    # Classifier
    x = Dense(512, name='fc1')(x)
    x = BatchNormalization(name='bn3')(x)
    x = Activation('relu', name='relu3')(x)
    x = Dropout(0.4, name='drop5')(x)

    x = Dense(256, name='fc2')(x)
    x = BatchNormalization(name='bn4')(x)
    x = Activation('relu', name='relu4')(x)
    x = Dropout(0.3, name='drop6')(x)

    outputs = Dense(n_classes, activation='softmax', name='output')(x)

    model_v2 = Model(inputs=inputs, outputs=outputs,
                     name='CNN_BiLSTM_Attention_V2')
    return model_v2

model_v2 = build_model_v2(n_features, n_classes)
model_v2.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-3,
        weight_decay=1e-4
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model_v2.summary()

# ── Architecture Plot V2 ──────────────────────────────────
layers_info_v2 = [
    ('Input',            '(batch, 65, 1)',                    '#E6F1FB', '#185FA5'),
    ('Conv1D',           '128 filters, kernel=3, padding=same','#EAF3DE', '#3B6D11'),
    ('Conv1D',           '256 filters, kernel=3, padding=same','#EAF3DE', '#3B6D11'),
    ('BatchNorm + ReLU', 'Dropout 0.2',                       '#EAF3DE', '#3B6D11'),
    ('MaxPooling1D',     'pool=2  →  (batch, 32, 256)',       '#EAF3DE', '#3B6D11'),
    ('BiLSTM',           '256 units × 2 dirs → 512',          '#EEEDFE', '#3C3489'),
    ('BiLSTM',           '128 units × 2 dirs → 256',          '#EEEDFE', '#3C3489'),
    ('Dropout 0.3',      '',                                   '#EEEDFE', '#3C3489'),
    ('Attention',        'Softmax weighted',                   '#FAEEDA', '#633806'),
    ('GlobalAvgPool1D',  '(batch, 256)',                       '#FAEEDA', '#633806'),
    ('Dense 512',        'BatchNorm + ReLU + Dropout 0.4',    '#F1EFE8', '#444441'),
    ('Dense 256',        'BatchNorm + ReLU + Dropout 0.3',    '#F1EFE8', '#444441'),
    ('Output',           'Dense 9 + Softmax',                 '#FCEBEB', '#791F1F'),
]

fig, ax = plt.subplots(figsize=(10, 14))
ax.axis('off')

box_w, box_h, gap = 0.55, 0.052, 0.014
x0, y = 0.5, 0.97

for i, (name, desc, fc, tc) in enumerate(layers_info_v2):
    y -= (box_h + gap)
    patch = mpatches.FancyBboxPatch(
        (x0 - box_w/2, y), box_w, box_h,
        boxstyle="round,pad=0.01",
        facecolor=fc, edgecolor=tc, linewidth=1.2,
        transform=ax.transAxes, clip_on=False
    )
    ax.add_patch(patch)
    ax.text(x0, y + box_h * 0.65, name,
            ha='center', va='center', fontsize=11,
            fontweight='bold', color=tc, transform=ax.transAxes)
    if desc:
        ax.text(x0, y + box_h * 0.22, desc,
                ha='center', va='center', fontsize=8.5,
                color=tc, alpha=0.85, transform=ax.transAxes)
    if i < len(layers_info_v2) - 1:
        ax.annotate('',
                    xy=(x0, y - 0.003),
                    xytext=(x0, y - gap + 0.003),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='#888780', lw=1.5))

total_params_v2 = sum([p.numpy().size for p in model_v2.trainable_variables])
ax.set_title(
    f'CNN + BiLSTM + Attention V2 — IDS Architecture\nTotal trainable params: {total_params_v2:,}',
    fontsize=13, fontweight='bold', pad=12, color='#2C2C2A'
)
plt.tight_layout()
plt.savefig(MODEL_PATH_V2 + 'architecture_v2.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()
print("\n✅ Model V2 Built & Architecture Saved")

Model: "CNN_BiLSTM_Attention_V2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 65, 1)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1 (Conv1D)      │ (None, 65, 128)   │        512 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn1                 │ (None, 65, 128)   │        512 │ conv1[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu1 (Activation)  │ (None, 65, 128)   │          0 │ bn1[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop1 (Dropout)     │ (None, 65, 128)   │          0 │ relu1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2 (Conv1D)      │ (None, 65, 256)   │     98,560 │ drop1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn2                 │ (None, 65, 256)   │      1,024 │ conv2[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu2 (Activation)  │ (None, 65, 256)   │          0 │ bn2[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop2 (Dropout)     │ (None, 65, 256)   │          0 │ relu2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 32, 256)   │          0 │ drop2[0][0]       │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm1             │ (None, 32, 512)   │  1,050,624 │ pool1[0][0]       │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop3 (Dropout)     │ (None, 32, 512)   │          0 │ bilstm1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm2             │ (None, 32, 256)   │    656,384 │ drop3[0][0]       │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop4 (Dropout)     │ (None, 32, 256)   │          0 │ bilstm2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32, 1)     │        257 │ drop4[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 32)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 32)        │          0 │ flatten_1[0][0]   │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector_1     │ (None, 256, 32)   │          0 │ activation_1[0][… │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ permute_1 (Permute) │ (None, 32, 256)   │          0 │ repeat_vector_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 32, 256)   │          0 │ drop4[0][0],      │
│ (Multiply)          │                   │            │ permute_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap                 │ (None, 256)       │          0 │ multiply_1[0][0]  │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc1 (Dense)         │ (None, 512)       │    131,584 │ gap[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn3                 │ (None, 512)       │      2,048 │ fc1[0][0]         │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu3 (Activation)  │ (None, 512)       │          0 │ bn3[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop5 (Dropout)     │ (None, 512)       │          0 │ relu3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc2 (Dense)         │ (None, 256)       │    131,328 │ drop5[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn4                 │ (None, 256)       │      1,024 │ fc2[0][0]         │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu4 (Activation)  │ (None, 256)       │          0 │ bn4[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop6 (Dropout)     │ (None, 256)       │          0 │ relu4[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 9)         │      2,313 │ drop6[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,076,170 (7.92 MB)

 Trainable params: 2,073,866 (7.91 MB)

 Non-trainable params: 2,304 (9.00 KB)



# ║      CELL 9 — Train V2                                  ║


In [ ]:
 ╔══════════════════════════════════════════════════════════╗
# ║      CELL 9 — Train V2                                  ║
# ╚══════════════════════════════════════════════════════════╝

callbacks_v2 = [
    EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=MODEL_PATH_V2 + 'best_model_v2.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

print("Training V2 started...")
history_v2 = model_v2.fit(
    X_train_3d, y_train,
    validation_data=(X_val_3d, y_val),
    epochs=100,
    batch_size=2048,
    callbacks=callbacks_v2,
    class_weight=class_weight_dict,
    verbose=1
)

model_v2.save(MODEL_PATH_V2 + 'final_model_v2.keras')
print("\n✅ V2 Training Done & Model Saved")

# ── Training Curves V2 ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_v2.history['accuracy'],
             color='#185FA5', linewidth=2, label='Train Accuracy')
axes[0].plot(history_v2.history['val_accuracy'],
             color='#D85A30', linewidth=2, linestyle='--', label='Val Accuracy')
axes[0].set_title('V2 — Model Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1])

axes[1].plot(history_v2.history['loss'],
             color='#185FA5', linewidth=2, label='Train Loss')
axes[1].plot(history_v2.history['val_loss'],
             color='#D85A30', linewidth=2, linestyle='--', label='Val Loss')
axes[1].set_title('V2 — Model Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Loss', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

best_epoch_v2 = np.argmin(history_v2.history['val_loss'])
for ax in axes:
    ax.axvline(x=best_epoch_v2, color='#3B6D11', linewidth=1.5,
               linestyle=':', label=f'Best epoch ({best_epoch_v2+1})')
    ax.legend(fontsize=10)

plt.suptitle('CNN + BiLSTM + Attention V2 — Training History',
             fontsize=14, fontweight='bold', color='#2C2C2A', y=1.02)
plt.tight_layout()
plt.savefig(MODEL_PATH_V2 + 'training_curves_v2.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()

best_val_acc_v2  = max(history_v2.history['val_accuracy'])
best_val_loss_v2 = min(history_v2.history['val_loss'])
total_epochs_v2  = len(history_v2.history['loss'])

print(f"\n=== V2 Training Summary ===")
print(f"Total epochs ran : {total_epochs_v2}")
print(f"Best epoch       : {best_epoch_v2 + 1}")
print(f"Best val accuracy: {best_val_acc_v2*100:.2f}%")
print(f"Best val loss    : {best_val_loss_v2:.4f}")

Training V2 started...
Epoch 1/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 401ms/step - accuracy: 0.9219 - loss: 0.2486
Epoch 1: val_loss improved from None to 0.02756, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 1: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 270s 430ms/step - accuracy: 0.9743 - loss: 0.0842 - val_accuracy: 0.9924 - val_loss: 0.0276 - learning_rate: 0.0010
Epoch 2/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9925 - loss: 0.0228
Epoch 2: val_loss improved from 0.02756 to 0.01798, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 2: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9929 - loss: 0.0215 - val_accuracy: 0.9938 - val_loss: 0.0180 - learning_rate: 0.0010
Epoch 3/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9933 - loss: 0.0196
Epoch 3: val_loss improved from 0.01798 to 0.01729, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 3: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9935 - loss: 0.0190 - val_accuracy: 0.9941 - val_loss: 0.0173 - learning_rate: 0.0010
Epoch 4/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9937 - loss: 0.0181
Epoch 4: val_loss improved from 0.01729 to 0.01670, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 4: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 260s 425ms/step - accuracy: 0.9939 - loss: 0.0178 - val_accuracy: 0.9942 - val_loss: 0.0167 - learning_rate: 0.0010
Epoch 5/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9939 - loss: 0.0175
Epoch 5: val_loss improved from 0.01670 to 0.01621, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 5: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 260s 425ms/step - accuracy: 0.9940 - loss: 0.0172 - val_accuracy: 0.9942 - val_loss: 0.0162 - learning_rate: 0.0010
Epoch 6/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9941 - loss: 0.0164
Epoch 6: val_loss improved from 0.01621 to 0.01559, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 6: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9943 - loss: 0.0160 - val_accuracy: 0.9942 - val_loss: 0.0156 - learning_rate: 0.0010
Epoch 7/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9943 - loss: 0.0159
Epoch 7: val_loss improved from 0.01559 to 0.01539, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 7: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9944 - loss: 0.0156 - val_accuracy: 0.9943 - val_loss: 0.0154 - learning_rate: 0.0010
Epoch 8/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9942 - loss: 0.0160
Epoch 8: val_loss did not improve from 0.01539
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9940 - loss: 0.0178 - val_accuracy: 0.9786 - val_loss: 0.0660 - learning_rate: 0.0010
Epoch 9/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9925 - loss: 0.0217
Epoch 9: val_loss improved from 0.01539 to 0.01505, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 9: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9939 - loss: 0.0174 - val_accuracy: 0.9944 - val_loss: 0.0151 - learning_rate: 0.0010
Epoch 10/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9944 - loss: 0.0153
Epoch 10: val_loss improved from 0.01505 to 0.01460, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 10: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9945 - loss: 0.0149 - val_accuracy: 0.9945 - val_loss: 0.0146 - learning_rate: 0.0010
Epoch 11/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9945 - loss: 0.0149
Epoch 11: val_loss did not improve from 0.01460
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9947 - loss: 0.0146 - val_accuracy: 0.9941 - val_loss: 0.0149 - learning_rate: 0.0010
Epoch 12/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9945 - loss: 0.0147
Epoch 12: val_loss improved from 0.01460 to 0.01378, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 12: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9947 - loss: 0.0141 - val_accuracy: 0.9945 - val_loss: 0.0138 - learning_rate: 0.0010
Epoch 13/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9945 - loss: 0.0142
Epoch 13: val_loss improved from 0.01378 to 0.01363, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 13: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9947 - loss: 0.0137 - val_accuracy: 0.9952 - val_loss: 0.0136 - learning_rate: 0.0010
Epoch 14/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9947 - loss: 0.0141
Epoch 14: val_loss improved from 0.01363 to 0.01319, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 14: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9948 - loss: 0.0137 - val_accuracy: 0.9948 - val_loss: 0.0132 - learning_rate: 0.0010
Epoch 15/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9948 - loss: 0.0134
Epoch 15: val_loss did not improve from 0.01319
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9949 - loss: 0.0131 - val_accuracy: 0.9945 - val_loss: 0.0135 - learning_rate: 0.0010
Epoch 16/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9949 - loss: 0.0134
Epoch 16: val_loss improved from 0.01319 to 0.01279, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 16: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9951 - loss: 0.0130 - val_accuracy: 0.9950 - val_loss: 0.0128 - learning_rate: 0.0010
Epoch 17/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9950 - loss: 0.0133
Epoch 17: val_loss improved from 0.01279 to 0.01265, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 17: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9951 - loss: 0.0130 - val_accuracy: 0.9952 - val_loss: 0.0127 - learning_rate: 0.0010
Epoch 18/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9951 - loss: 0.0129
Epoch 18: val_loss did not improve from 0.01265
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9952 - loss: 0.0126 - val_accuracy: 0.9953 - val_loss: 0.0127 - learning_rate: 0.0010
Epoch 19/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9951 - loss: 0.0128
Epoch 19: val_loss improved from 0.01265 to 0.01244, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 19: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9952 - loss: 0.0125 - val_accuracy: 0.9953 - val_loss: 0.0124 - learning_rate: 0.0010
Epoch 20/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9953 - loss: 0.0124
Epoch 20: val_loss improved from 0.01244 to 0.01144, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 20: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9954 - loss: 0.0121 - val_accuracy: 0.9957 - val_loss: 0.0114 - learning_rate: 0.0010
Epoch 21/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9954 - loss: 0.0120
Epoch 21: val_loss did not improve from 0.01144
613/613 ━━━━━━━━━━━━━━━━━━━━ 260s 424ms/step - accuracy: 0.9956 - loss: 0.0116 - val_accuracy: 0.9954 - val_loss: 0.0117 - learning_rate: 0.0010
Epoch 22/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9955 - loss: 0.0117
Epoch 22: val_loss improved from 0.01144 to 0.01054, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 22: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9956 - loss: 0.0114 - val_accuracy: 0.9962 - val_loss: 0.0105 - learning_rate: 0.0010
Epoch 23/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9957 - loss: 0.0115
Epoch 23: val_loss did not improve from 0.01054
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9958 - loss: 0.0113 - val_accuracy: 0.9959 - val_loss: 0.0110 - learning_rate: 0.0010
Epoch 24/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9957 - loss: 0.0111
Epoch 24: val_loss did not improve from 0.01054
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9958 - loss: 0.0110 - val_accuracy: 0.9959 - val_loss: 0.0106 - learning_rate: 0.0010
Epoch 25/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9959 - loss: 0.0110
Epoch 25: val_loss improved from 0.01054 to 0.01030, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 25: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9960 - loss: 0.0107 - val_accuracy: 0.9960 - val_loss: 0.0103 - learning_rate: 0.0010
Epoch 26/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9960 - loss: 0.0106
Epoch 26: val_loss did not improve from 0.01030
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9960 - loss: 0.0105 - val_accuracy: 0.9958 - val_loss: 0.0105 - learning_rate: 0.0010
Epoch 27/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9961 - loss: 0.0105
Epoch 27: val_loss improved from 0.01030 to 0.01028, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 27: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9961 - loss: 0.0103 - val_accuracy: 0.9960 - val_loss: 0.0103 - learning_rate: 0.0010
Epoch 28/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 397ms/step - accuracy: 0.9960 - loss: 0.0106
Epoch 28: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 28: val_loss did not improve from 0.01028
613/613 ━━━━━━━━━━━━━━━━━━━━ 260s 424ms/step - accuracy: 0.9960 - loss: 0.0107 - val_accuracy: 0.9962 - val_loss: 0.0108 - learning_rate: 0.0010
Epoch 29/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9961 - loss: 0.0104
Epoch 29: val_loss improved from 0.01028 to 0.00960, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 29: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9963 - loss: 0.0098 - val_accuracy: 0.9966 - val_loss: 0.0096 - learning_rate: 5.0000e-04
Epoch 30/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9965 - loss: 0.0094
Epoch 30: val_loss improved from 0.00960 to 0.00931, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 30: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9965 - loss: 0.0092 - val_accuracy: 0.9966 - val_loss: 0.0093 - learning_rate: 5.0000e-04
Epoch 31/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9965 - loss: 0.0092
Epoch 31: val_loss did not improve from 0.00931
613/613 ━━━━━━━━━━━━━━━━━━━━ 260s 425ms/step - accuracy: 0.9966 - loss: 0.0090 - val_accuracy: 0.9965 - val_loss: 0.0095 - learning_rate: 5.0000e-04
Epoch 32/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9965 - loss: 0.0092
Epoch 32: val_loss did not improve from 0.00931
613/613 ━━━━━━━━━━━━━━━━━━━━ 260s 425ms/step - accuracy: 0.9966 - loss: 0.0090 - val_accuracy: 0.9965 - val_loss: 0.0094 - learning_rate: 5.0000e-04
Epoch 33/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9966 - loss: 0.0091
Epoch 33: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 33: val_loss improved from 0.00931 to 0.00926, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 33: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9967 - loss: 0.0089 - val_accuracy: 0.9966 - val_loss: 0.0093 - learning_rate: 5.0000e-04
Epoch 34/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9967 - loss: 0.0087
Epoch 34: val_loss improved from 0.00926 to 0.00916, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 34: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9968 - loss: 0.0084 - val_accuracy: 0.9965 - val_loss: 0.0092 - learning_rate: 2.5000e-04
Epoch 35/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9967 - loss: 0.0086
Epoch 35: val_loss did not improve from 0.00916
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9968 - loss: 0.0084 - val_accuracy: 0.9965 - val_loss: 0.0092 - learning_rate: 2.5000e-04
Epoch 36/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9968 - loss: 0.0085
Epoch 36: val_loss improved from 0.00916 to 0.00904, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 36: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9969 - loss: 0.0083 - val_accuracy: 0.9966 - val_loss: 0.0090 - learning_rate: 2.5000e-04
Epoch 37/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9968 - loss: 0.0084
Epoch 37: val_loss did not improve from 0.00904
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9969 - loss: 0.0082 - val_accuracy: 0.9966 - val_loss: 0.0091 - learning_rate: 2.5000e-04
Epoch 38/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9967 - loss: 0.0084
Epoch 38: val_loss did not improve from 0.00904
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9968 - loss: 0.0083 - val_accuracy: 0.9966 - val_loss: 0.0091 - learning_rate: 2.5000e-04
Epoch 39/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9968 - loss: 0.0084
Epoch 39: val_loss improved from 0.00904 to 0.00894, saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras

Epoch 39: finished saving model to /kaggle/working/CNN_Model_V2/best_model_v2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9969 - loss: 0.0083 - val_accuracy: 0.9967 - val_loss: 0.0089 - learning_rate: 2.5000e-04
Epoch 40/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9968 - loss: 0.0083
Epoch 40: val_loss did not improve from 0.00894
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9969 - loss: 0.0081 - val_accuracy: 0.9967 - val_loss: 0.0090 - learning_rate: 2.5000e-04
Epoch 41/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9968 - loss: 0.0084
Epoch 41: val_loss did not improve from 0.00894
613/613 ━━━━━━━━━━━━━━━━━━━━ 260s 425ms/step - accuracy: 0.9969 - loss: 0.0082 - val_accuracy: 0.9966 - val_loss: 0.0090 - learning_rate: 2.5000e-04
Epoch 42/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9968 - loss: 0.0083
Epoch 42: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 42: val_loss did not improve from 0.00894
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9969 - loss: 0.0081 - val_accuracy: 0.9966 - val_loss: 0.0091 - learning_rate: 2.5000e-04
Epoch 43/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9968 - loss: 0.0082
Epoch 43: val_loss did not improve from 0.00894
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9970 - loss: 0.0080 - val_accuracy: 0.9966 - val_loss: 0.0090 - learning_rate: 1.2500e-04
Epoch 44/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9969 - loss: 0.0081
Epoch 44: val_loss did not improve from 0.00894
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 426ms/step - accuracy: 0.9970 - loss: 0.0079 - val_accuracy: 0.9967 - val_loss: 0.0090 - learning_rate: 1.2500e-04
Epoch 45/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9969 - loss: 0.0081
Epoch 45: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.

Epoch 45: val_loss did not improve from 0.00894
613/613 ━━━━━━━━━━━━━━━━━━━━ 261s 425ms/step - accuracy: 0.9970 - loss: 0.0079 - val_accuracy: 0.9967 - val_loss: 0.0090 - learning_rate: 1.2500e-04
Epoch 46/100
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9969 - loss: 0.0080
Epoch 46: val_loss did not improve from 0.00894
613/613 ━━━━━━━━━━━━━━━━━━━━ 260s 425ms/step - accuracy: 0.9970 - loss: 0.0078 - val_accuracy: 0.9967 - val_loss: 0.0090 - learning_rate: 6.2500e-05
Epoch 46: early stopping
Restoring model weights from the end of the best epoch: 39.

✅ V2 Training Done & Model Saved



In [ ]:
=== V2 Training Summary ===
Total epochs ran : 46
Best epoch       : 39
Best val accuracy: 99.67%
Best val loss    : 0.0089

In [ ]:
 ╔══════════════════════════════════════════════════════════╗
# ║      CELL 12 — Evaluate V2                               ║
# ╚══════════════════════════════════════════════════════════╝

# ── Predict V2 ────────────────────────────────────────────
print("Predicting V2 on test set...")
y_pred_v2_prob = model_v2.predict(X_test_3d, batch_size=2048, verbose=1)
y_pred_v2      = np.argmax(y_pred_v2_prob, axis=1)

# ── Classification Report ─────────────────────────────────
print("\n=== Classification Report V2 ===")
report_v2 = classification_report(
    y_test, y_pred_v2,
    target_names=le.classes_,
    digits=4
)
print(report_v2)
with open(MODEL_PATH_V2 + 'classification_report_v2.txt', 'w') as f:
    f.write(report_v2)

# ── Overall Metrics ───────────────────────────────────────
acc_v2         = accuracy_score(y_test, y_pred_v2)
f1_macro_v2    = f1_score(y_test, y_pred_v2, average='macro')
f1_weighted_v2 = f1_score(y_test, y_pred_v2, average='weighted')
f1_per_class_v2 = f1_score(y_test, y_pred_v2, average=None)

print(f"Test Accuracy        : {acc_v2*100:.4f}%")
print(f"F1 Score (macro)     : {f1_macro_v2:.4f}")
print(f"F1 Score (weighted)  : {f1_weighted_v2:.4f}")

# ── Confusion Matrix ──────────────────────────────────────
cm_v2     = confusion_matrix(y_test, y_pred_v2)
cm_pct_v2 = cm_v2.astype('float') / cm_v2.sum(axis=1)[:, np.newaxis] * 100

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    cm_pct_v2, annot=True, fmt='.1f', cmap='Blues',
    xticklabels=le.classes_, yticklabels=le.classes_,
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Percentage (%)'}
)
ax.set_xlabel('Predicted Label', fontsize=12, labelpad=10)
ax.set_ylabel('True Label', fontsize=12, labelpad=10)
ax.set_title('CNN + BiLSTM + Attention V2 — Confusion Matrix (%)\nTest Set',
             fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=30, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig(MODEL_PATH_V2 + 'confusion_matrix_v2.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()

# ── Per-Class F1 Bar Plot ─────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#185FA5' if f >= 0.99 else '#D85A30' if f < 0.95 else '#BA7517'
          for f in f1_per_class_v2]
bars = ax.bar(le.classes_, f1_per_class_v2 * 100,
              color=colors, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, f1_per_class_v2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.2,
            f'{val*100:.2f}%',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylim([90, 101])
ax.set_ylabel('F1 Score (%)', fontsize=11)
ax.set_title('V2 — F1 Score per Class — Test Set',
             fontsize=13, fontweight='bold')
ax.axhline(y=99, color='#3B6D11', linewidth=1.5,
           linestyle='--', label='99% threshold')
ax.legend(fontsize=10)
plt.xticks(rotation=30, ha='right', fontsize=10)
plt.tight_layout()
plt.savefig(MODEL_PATH_V2 + 'f1_per_class_v2.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()

print("\n✅ Evaluation V2 Done")

Predicting V2 on test set...
132/132 ━━━━━━━━━━━━━━━━━━━━ 17s 127ms/step

=== Classification Report V2 ===
                        precision    recall  f1-score   support

                Benign     0.9993    0.9988    0.9991     30000
                   Bot     0.9998    0.9998    0.9998     30000
      DDoS attack-HOIC     1.0000    1.0000    1.0000     30000
DDoS attacks-LOIC-HTTP     0.9995    0.9999    0.9997     30000
           DoS attacks     0.9998    0.9999    0.9998     30000
        FTP-BruteForce     0.9999    1.0000    0.9999     29004
        Reconnaissance     0.9994    0.9866    0.9930     30000
         SQL Injection     0.9865    0.9936    0.9901     30000
        SSH-Bruteforce     0.9882    0.9936    0.9909     30000

              accuracy                         0.9969    269004
             macro avg     0.9969    0.9969    0.9969    269004
          weighted avg     0.9969    0.9969    0.9969    269004

Test Accuracy        : 99.6907%
F1 Score (macro)     : 0.9969
F1 Score (weighted)  : 0.9969

